<a href="https://colab.research.google.com/github/kalingasajja/FAS-deepfake-capstone/blob/Rohith/Capstone_final(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Connecting to drive

In [4]:
# Use --no-deps to bypass the Pillow version conflict
!pip install --no-deps facenet-pytorch

from google.colab import drive
import os
import cv2
import torch
import numpy as np
from facenet_pytorch import MTCNN
from PIL import Image

# Mount Google Drive to access your raw_videos folder
drive.mount('/content/drive')

# Set device to GPU if you changed your runtime type, otherwise CPU
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Initialize MTCNN
# keep_all=False: If multiple faces are detected, keep only the most prominent one
# margin=30: Adds a 30-pixel padding around the bounding box. This is crucial for deepfake detection
# so we capture the forehead and jawline blending edges, not just the inner face.
mtcnn = MTCNN(keep_all=False, margin=30, device=device)

print(f"✅ Drive mounted and MTCNN initialized on {device}")

  Using cached facenet_pytorch-2.6.0-py3-none-any.whl.metadata (12 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 16.7 MB/s  0:00:00
Mounted at /content/drive
✅ Drive mounted and MTCNN initialized on cuda:0


Extracting frames and cropping faces using MTCNN

In [5]:
import json
from tqdm.notebook import tqdm

# Set up paths for the new face-cropped dataset
BASE_DIR = '/content/drive/MyDrive/Deepfake_Capstone_Spatial'
DATASET_DIR = os.path.join(BASE_DIR, 'raw_videos')
MANIFEST_PATH = os.path.join(BASE_DIR, 'manifests', 'spatial_manifest.json')
FACES_DIR = os.path.join(BASE_DIR, 'extracted_faces') # New folder!

NUM_FRAMES = 10

# Load the manifest
with open(MANIFEST_PATH, 'r') as f:
    manifest = json.load(f)

def extract_faces_from_video(video_path, output_dir, video_id):
    """Reads a video, detects faces in 10 evenly spaced frames, and saves crops."""
    if not os.path.exists(video_path):
        return

    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        cap.release()
        return

    frame_indices = np.linspace(0, total_frames - 1, NUM_FRAMES, dtype=int)

    for idx, frame_idx in enumerate(frame_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()

        if ret:
            # OpenCV uses BGR, but MTCNN expects standard RGB
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img_pil = Image.fromarray(frame_rgb)

            # Construct the save path
            save_path = os.path.join(output_dir, f"{video_id}_face_{idx}.jpg")

            # mtcnn does the heavy lifting: detects, crops with margin, and saves!
            # We wrap it in a try-except just in case it encounters a corrupted frame
            try:
                mtcnn(img_pil, save_path=save_path)
            except Exception as e:
                pass

    cap.release()

# Process each split (train, val, test)
for split in ['train', 'val', 'test']:
    print(f"\nProcessing '{split}' split...")

    split_dir = os.path.join(FACES_DIR, split)
    real_dir = os.path.join(split_dir, '0')
    fake_dir = os.path.join(split_dir, '1')

    os.makedirs(real_dir, exist_ok=True)
    os.makedirs(fake_dir, exist_ok=True)

    split_data = manifest.get(split, [])

    for item in tqdm(split_data, desc=f"Extracting {split}"):
        video_id = item['video_id']
        label = item['label']

        out_dir = fake_dir if label == 1 else real_dir
        full_video_path = os.path.join(DATASET_DIR, item['path'])

        extract_faces_from_video(full_video_path, out_dir, video_id)

print("\nFace extraction complete! Your cropped dataset is ready.")


Processing 'train' split...


Extracting train:   0%|          | 0/346 [00:00<?, ?it/s]


Processing 'val' split...


Extracting val:   0%|          | 0/87 [00:00<?, ?it/s]


Processing 'test' split...


Extracting test:   0%|          | 0/67 [00:00<?, ?it/s]


Face extraction complete! Your cropped dataset is ready.


Loading Cropped Faces & Unfreezing SigLIP Layers

In [6]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from transformers import SiglipVisionModel

# 1. Data Transformations
siglip_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# 2. Setup DataLoaders for the NEW cropped faces
BASE_DIR = '/content/drive/MyDrive/Deepfake_Capstone_Spatial/extracted_faces'

train_dataset = datasets.ImageFolder(root=f'{BASE_DIR}/train', transform=siglip_transform)
val_dataset = datasets.ImageFolder(root=f'{BASE_DIR}/val', transform=siglip_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Loaded {len(train_dataset)} training faces, {len(val_dataset)} validation faces.")

# 3. Model Architecture with Partial Unfreezing
class FaceSpatialDetector(nn.Module):
    def __init__(self):
        super(FaceSpatialDetector, self).__init__()
        self.siglip = SiglipVisionModel.from_pretrained("google/siglip-base-patch16-224")

        # First, freeze the entire backbone
        for param in self.siglip.parameters():
            param.requires_grad = False

        # UNFREEZE the last transformer layer to learn face-specific deepfake artifacts
        for param in self.siglip.encoder.layers[-1].parameters():
            param.requires_grad = True

        hidden_size = self.siglip.config.hidden_size

        # 3-Layer MLP classification head
        self.mlp = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 1)
        )

    def forward(self, pixel_values):
        outputs = self.siglip(pixel_values=pixel_values)
        image_embeds = outputs.pooler_output
        logits = self.mlp(image_embeds)
        return logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FaceSpatialDetector().to(device)

print(f"Model initialized on: {device}")

Loaded 3442 training faces, 869 validation faces.


config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  813MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

[transformers] SiglipVisionModel LOAD REPORT from: google/siglip-base-patch16-224
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.lay

Model initialized on: cuda


Training the last layer of SigLIP Model and complete MLP with Differential Learning Rates

In [7]:
import torch.optim as optim
from torch.nn import BCEWithLogitsLoss
import copy
import os

# Hyperparameters
EPOCHS = 10

# Parameter groups for different learning rates
optimizer = optim.AdamW([
    {'params': model.siglip.encoder.layers[-1].parameters(), 'lr': 1e-5},
    {'params': model.mlp.parameters(), 'lr': 1e-3}
], weight_decay=1e-4)

criterion = BCEWithLogitsLoss()

best_val_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())

# Make sure the checkpoints directory exists
CHECKPOINT_DIR = '/content/drive/MyDrive/Deepfake_Capstone_Spatial/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, 'best_face_spatial_model.pth')

print("Starting Training Loop for Face-Cropped Model...")

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 20)

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
            dataloader = train_loader
        else:
            model.eval()
            dataloader = val_loader

        running_loss = 0.0
        running_corrects = 0
        total_samples = 0

        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.unsqueeze(1).float().to(device)

            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                logits = model(inputs)
                loss = criterion(logits, labels)
                preds = torch.sigmoid(logits) > 0.5

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            total_samples += inputs.size(0)

        epoch_loss = running_loss / total_samples
        epoch_acc = running_corrects.double() / total_samples

        print(f"{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

        if phase == 'val' and epoch_acc > best_val_acc:
            best_val_acc = epoch_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), CHECKPOINT_PATH)
            print(f"🌟 New best model saved with Val Acc: {best_val_acc:.4f}")

print("\nTraining complete!")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")

# Load the best weights back into the model for testing
model.load_state_dict(best_model_wts)

Starting Training Loop for Face-Cropped Model...

Epoch 1/10
--------------------
Train Loss: 0.1114 Acc: 0.9712
Val Loss: 0.2566 Acc: 0.9344
🌟 New best model saved with Val Acc: 0.9344

Epoch 2/10
--------------------
Train Loss: 0.0194 Acc: 0.9968
Val Loss: 0.3679 Acc: 0.9229

Epoch 3/10
--------------------
Train Loss: 0.0140 Acc: 0.9974
Val Loss: 0.1967 Acc: 0.9448
🌟 New best model saved with Val Acc: 0.9448

Epoch 4/10
--------------------
Train Loss: 0.0115 Acc: 0.9965
Val Loss: 0.2693 Acc: 0.9229

Epoch 5/10
--------------------
Train Loss: 0.0043 Acc: 0.9994
Val Loss: 0.3357 Acc: 0.9321

Epoch 6/10
--------------------
Train Loss: 0.0021 Acc: 1.0000
Val Loss: 0.3981 Acc: 0.9241

Epoch 7/10
--------------------
Train Loss: 0.0018 Acc: 1.0000
Val Loss: 0.3832 Acc: 0.9344

Epoch 8/10
--------------------
Train Loss: 0.0011 Acc: 1.0000
Val Loss: 0.3489 Acc: 0.9367

Epoch 9/10
--------------------
Train Loss: 0.0021 Acc: 0.9994
Val Loss: 0.3463 Acc: 0.9356

Epoch 10/10
-------------

<All keys matched successfully>

Video-Level Evaluation & Metric Calculation

In [8]:
import glob
import torch
import os
import json
import numpy as np
from PIL import Image
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

model.eval()
test_dir = '/content/drive/MyDrive/Deepfake_Capstone_Spatial/extracted_faces/test'
PREDS_DIR = '/content/drive/MyDrive/Deepfake_Capstone_Spatial/predictions'
os.makedirs(PREDS_DIR, exist_ok=True)

video_probs = {}

# Find all cropped test images
test_images = glob.glob(f'{test_dir}/*/*.jpg')

print(f"Found {len(test_images)} test face frames. Running inference...")

for img_path in test_images:
    filename = os.path.basename(img_path)
    # Extract video_id (e.g., 'face_swap_Kaggle_015_face_3.jpg' -> 'face_swap_Kaggle_015')
    video_id = filename.split('_face_')[0]

    img = Image.open(img_path).convert('RGB')
    input_tensor = siglip_transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_tensor)
        prob = torch.sigmoid(logits).item()

    if video_id not in video_probs:
        video_probs[video_id] = []
    video_probs[video_id].append(prob)

# Aggregate video-level predictions
y_true = []
y_prob = []
y_pred = []
video_level_preds = {}

for video_id, probs in video_probs.items():
    avg_prob = np.mean(probs)
    video_level_preds[video_id] = avg_prob

    true_label = 0 if video_id.startswith('real') else 1
    pred_label = 1 if avg_prob >= 0.5 else 0

    y_true.append(true_label)
    y_prob.append(avg_prob)
    y_pred.append(pred_label)

# Save the new predictions
output_path = os.path.join(PREDS_DIR, 'cropped_spatial_test_preds.json')
with open(output_path, 'w') as f:
    json.dump(video_level_preds, f, indent=4)

# Calculate Metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
auc_roc = roc_auc_score(y_true, y_prob)
conf_matrix = confusion_matrix(y_true, y_pred)

print("\n--- Cropped Spatial Branch Evaluation Metrics ---")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"AUC-ROC:   {auc_roc:.4f}")
print("\nConfusion Matrix (TN, FP | FN, TP):")
print(conf_matrix)

# Break down accuracy by category
categories = {'synthetic': {'correct': 0, 'total': 0},
              'face_swap': {'correct': 0, 'total': 0},
              'real': {'correct': 0, 'total': 0}}

for video_id, is_pred_fake in zip(video_probs.keys(), y_pred):
    true_label = 0 if video_id.startswith('real') else 1
    is_correct = (is_pred_fake == true_label)

    if video_id.startswith('synthetic'):
        cat = 'synthetic'
    elif video_id.startswith('face_swap'):
        cat = 'face_swap'
    else:
        cat = 'real'

    categories[cat]['total'] += 1
    if is_correct:
        categories[cat]['correct'] += 1

print("\n--- Accuracy by Category ---")
for cat, stats in categories.items():
    if stats['total'] > 0:
        cat_acc = stats['correct'] / stats['total']
        print(f"{cat.capitalize()}: {cat_acc:.4f} ({stats['correct']}/{stats['total']})")

Found 666 test face frames. Running inference...

--- Cropped Spatial Branch Evaluation Metrics ---
Accuracy:  0.8806
Precision: 0.8387
Recall:    0.8966
F1-Score:  0.8667
AUC-ROC:   0.9619

Confusion Matrix (TN, FP | FN, TP):
[[33  5]
 [ 3 26]]

--- Accuracy by Category ---
Synthetic: 0.9474 (18/19)
Face_swap: 0.8000 (8/10)
Real: 0.8684 (33/38)
